In [1]:
import unicodedata
import pandas as pd

In [2]:
file_path = "data/starbucks_drinkMenu_expanded.csv"

df = None

try:
  df = pd.read_csv(file_path, encoding='utf-8')
  used_encoding = 'utf-8'
  print(f"-> Archivo cargado exitosamente usando: 'utf-8'")
except UnicodeDecodeError:
  print(f"-> Error al leer con 'utf-8'. Intentando con 'latin-1'...")

if df is None:
  raise ValueError("No se pudo leer el archivo con las codificaciones comunes.")

# Vista previa
print(f"Dimensiones: {df.shape}")
df.head(10)

-> Archivo cargado exitosamente usando: 'utf-8'
Dimensiones: (242, 18)


,Beverage_category,Beverage,Beverage_prep,Calories,Total Fat (g),Trans Fat (g),Saturated Fat (g),Sodium (mg),Total Carbohydrates (g),Cholesterol (mg),Dietary Fibre (g),Sugars (g),Protein (g),Vitamin A (% DV),Vitamin C (% DV),Calcium (% DV),Iron (% DV),Caffeine (mg)
0,Coffee,Brewed Coffee,Short,3,0.1,0.0,0.0,0,5,0,0,0,0.3,0%,0%,0%,0%,175
1,Coffee,Brewed Coffee,Tall,4,0.1,0.0,0.0,0,10,0,0,0,0.5,0%,0%,0%,0%,260
2,Coffee,Brewed Coffee,Grande,5,0.1,0.0,0.0,0,10,0,0,0,1.0,0%,0%,0%,0%,330
3,Coffee,Brewed Coffee,Venti,5,0.1,0.0,0.0,0,10,0,0,0,1.0,0%,0%,2%,0%,410
4,Classic Espresso Drinks,Caffè Latte,Short Nonfat Milk,70,0.1,0.1,0.0,5,75,10,0,9,6.0,10%,0%,20%,0%,75
5,Classic Espresso Drinks,Caffè Latte,2% Milk,100,3.5,2.0,0.1,15,85,10,0,9,6.0,10%,0%,20%,0%,75
6,Classic Espresso Drinks,Caffè Latte,Soymilk,70,2.5,0.4,0.0,0,65,6,1,4,5.0,6%,0%,20%,8%,75
7,Classic Espresso Drinks,Caffè Latte,Tall Nonfat Milk,100,0.2,0.2,0.0,5,120,15,0,14,10.0,15%,0%,30%,0%,75
8,Classic Espresso Drinks,Caffè Latte,2% Milk,150,6,3.0,0.2,25,135,15,0,14,10.0,15%,0%,30%,0%,75
9,Classic Espresso Drinks,Caffè Latte,Soymilk,110,4.5,0.5,0.0,0,105,10,1,6,8.0,10%,0%,30%,15%,75


In [3]:
def standardize_to_utf8(text):
  """Limpia espacios, normaliza tildes/caracteres especiales y asegura texto UTF-8."""
  if pd.isna(text):
    return text

  text = str(text)
  # Normalización canónica (compone acentos y caracteres especiales correctamente)
  text = unicodedata.normalize("NFC", text)
  # Asegurar bytes compatibles con UTF-8
  text = text.encode("utf-8", errors="replace").decode("utf-8")
  return text.strip()

In [4]:
# 1. Limpiar los nombres de las columnas
df.columns = [standardize_to_utf8(col) for col in df.columns]

# 2. Seleccionar columnas de tipo texto (object)
text_columns = df.select_dtypes(include=["object"]).columns
print(f"Columnas de texto procesadas: {list(text_columns)}")

for col in text_columns:
  df[col] = df[col].apply(standardize_to_utf8)

df.info()

Columnas de texto procesadas: ['Beverage_category', 'Beverage', 'Beverage_prep', 'Total Fat (g)', 'Vitamin A (% DV)', 'Vitamin C (% DV)', 'Calcium (% DV)', 'Iron (% DV)', 'Caffeine (mg)']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242 entries, 0 to 241
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Beverage_category        242 non-null    object 
 1   Beverage                 242 non-null    object 
 2   Beverage_prep            242 non-null    object 
 3   Calories                 242 non-null    int64  
 4   Total Fat (g)            242 non-null    object 
 5   Trans Fat (g)            242 non-null    float64
 6   Saturated Fat (g)        242 non-null    float64
 7   Sodium (mg)              242 non-null    int64  
 8   Total Carbohydrates (g)  242 non-null    int64  
 9   Cholesterol (mg)         242 non-null    int64  
 10  Dietary Fibre (g)        242 non-null    int64  
 11  

In [5]:
# cambiar varias a la vez
df = df.rename(columns={
    'Beverage_category': 'Categoria_bebida',
    'Beverage': 'Bebida',
    'Beverage_prep': 'Preparacion_bebida',
    'Calories': 'Calories (Cal)',
})

In [6]:
# Lista de columnas de porcentaje en este archivo (asegúrate de que coincida con tus nombres limpios)
columnas_pct = [
    'Vitamin A (% DV)',
    'Vitamin C (% DV)',
    'Calcium (% DV)',
    'Iron (% DV)',
]

for col in columnas_pct:
  if col in df.columns:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False).str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [7]:
df.head(10)

,Categoria_bebida,Bebida,Preparacion_bebida,Calories (Cal),Total Fat (g),Trans Fat (g),Saturated Fat (g),Sodium (mg),Total Carbohydrates (g),Cholesterol (mg),Dietary Fibre (g),Sugars (g),Protein (g),Vitamin A (% DV),Vitamin C (% DV),Calcium (% DV),Iron (% DV),Caffeine (mg)
0,Coffee,Brewed Coffee,Short,3,0.1,0.0,0.0,0,5,0,0,0,0.3,0,0,0,0.0,175
1,Coffee,Brewed Coffee,Tall,4,0.1,0.0,0.0,0,10,0,0,0,0.5,0,0,0,0.0,260
2,Coffee,Brewed Coffee,Grande,5,0.1,0.0,0.0,0,10,0,0,0,1.0,0,0,0,0.0,330
3,Coffee,Brewed Coffee,Venti,5,0.1,0.0,0.0,0,10,0,0,0,1.0,0,0,2,0.0,410
4,Classic Espresso Drinks,Caffè Latte,Short Nonfat Milk,70,0.1,0.1,0.0,5,75,10,0,9,6.0,10,0,20,0.0,75
5,Classic Espresso Drinks,Caffè Latte,2% Milk,100,3.5,2.0,0.1,15,85,10,0,9,6.0,10,0,20,0.0,75
6,Classic Espresso Drinks,Caffè Latte,Soymilk,70,2.5,0.4,0.0,0,65,6,1,4,5.0,6,0,20,8.0,75
7,Classic Espresso Drinks,Caffè Latte,Tall Nonfat Milk,100,0.2,0.2,0.0,5,120,15,0,14,10.0,15,0,30,0.0,75
8,Classic Espresso Drinks,Caffè Latte,2% Milk,150,6,3.0,0.2,25,135,15,0,14,10.0,15,0,30,0.0,75
9,Classic Espresso Drinks,Caffè Latte,Soymilk,110,4.5,0.5,0.0,0,105,10,1,6,8.0,10,0,30,15.0,75


In [8]:
output_path = "starbucks_drinkMenu_expanded_utf8.csv"

# Guardar asegurando UTF-8 explícito
df.to_csv(output_path, index=False, encoding="utf-8")
print(
    f"Dataset estandarizado guardado correctamente en UTF-8 como: '{output_path}'"
)

Dataset estandarizado guardado correctamente en UTF-8 como: 'starbucks_drinkMenu_expanded_utf8.csv'
